# Week 4 Exercise: Prompt-Based Solutions Using the OpenAI API

**Adjovi Gloria Lembila**  
**DSC670 Advanced Uses of Generative AI**  
**Professor Frank Neugebauer**  
**July 2026**

## Assignment Overview

This notebook completes the Week 4 exercise using Python, Jupyter Notebook, and the OpenAI API. The exercise has two parts. Problem 1 uses zero-shot and one-shot prompting to extract shoe order information from emails and return structured JSON. Problem 2 recreates the textbook-style chat completion example using OpenAI instead of Azure OpenAI.

My goal in this notebook is not only to produce working code, but also to evaluate the quality of the model responses. I explain what I expected the model to do, how the outputs should be checked, and how the responses compare across the two problems.

## Setup and API Configuration

The OpenAI API key should not be typed directly into the notebook. Instead, I use an environment variable called `OPENAI_API_KEY`. This is safer because the notebook can be submitted or shared without exposing the key.

Before running the notebook, the key can be set in a terminal or command prompt. On Windows PowerShell, the command is `setx OPENAI_API_KEY "your_api_key_here"`. On Mac or Linux, the command is `export OPENAI_API_KEY="your_api_key_here"`.

This setup matters because the assignment requires Python and an API. Using an API makes the work repeatable because the same prompts can be sent again and the results can be compared.

In [ ]:
# If needed, install the OpenAI package first:
# !pip install openai

import os
import json
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. Set it as an environment variable before running this notebook."
    )

client = OpenAI(api_key=api_key)

# You may change this model if your instructor or account requires a different one.
MODEL = "gpt-4o-mini"

# Problem 1: Math and Extraction With Zero-Shot and One-Shot Prompting

Problem 1 combines information extraction and math. The model must read an email, identify the agent name, identify the customer, separate the address fields, extract each shoe item, assign a price, calculate subtotals, and calculate the grand total.

The problem is useful because it tests more than simple extraction. The model must keep the JSON structure consistent while also doing arithmetic correctly. That means the result should be evaluated for both formatting and accuracy.

In [ ]:
email_1 = """
Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229
"""

email_2 = """
Hey Michelle,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 1 pair
2. Converse All-Star - 20 pair
3. New Balance 990 - 2 pair
4. Nike Zoom Fly 5 men's red - 5 pair

Please provide sub-totals and grand total cost.

Thanks.

Artis Gilmore
723 Lexington Blvd.
New York, NY 10001
(503) 484-1029
"""

shoe_prices = {
    "Nike Air Jordan I": 100,
    "Converse All-Star": 30,
    "New Balance 990": 110,
    "Nike Zoom Fly 5 men's red": 400
}

## Problem 1A: Zero-Shot Prompt

The zero-shot prompt gives the model the task instructions and the email, but it does not provide a completed example. This allows me to test whether the model can understand the extraction task from directions alone.

The benefit of zero-shot prompting is that it is direct and short. The weakness is that the model has more freedom to choose its own formatting. For this assignment, that matters because the output must be valid JSON. To reduce errors, I list every required field and tell the model to return only JSON.

In [ ]:
zero_shot_prompt = f"""
Extract all order information from the email below and return ONLY valid JSON.
Do not include Markdown, explanation, or code fences.

Required fields:
- shoe_agent_name
- request_type
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone
- items: a list of shoe objects containing shoe_brand, shoe_model, shoe_quantity, shoe_price, and shoe_subtotal
- grand_total

Use these shoe prices:
- Nike Air Jordan I: 100
- Converse All-Star: 30
- New Balance 990: 110
- Nike Zoom Fly 5 men's red: 400

The request_type should be "Order".
Calculate each shoe_subtotal as shoe_quantity multiplied by shoe_price.
Calculate grand_total as the sum of all shoe_subtotal values.

Email:
{email_1}
"""

zero_shot_response = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a careful data extraction assistant. Return valid JSON only."
        },
        {
            "role": "user",
            "content": zero_shot_prompt
        }
    ],
    temperature=0
)

zero_shot_json = json.loads(zero_shot_response.choices[0].message.content)
print(json.dumps(zero_shot_json, indent=2))

## Zero-Shot Result Commentary

For the first email, the expected shoe agent name is Jim because the email begins with “Hey Jim.” The expected customer is Lance Gentry. The request type should be “Order” because the customer is requesting that shoes be shipped.

The math should also be checked carefully. Nike Air Jordan I has a quantity of 2 and a price of 100, so the subtotal should be 200. Converse All-Star has a quantity of 10 and a price of 30, so the subtotal should be 300. New Balance 990 has a quantity of 1 and a price of 110, so the subtotal should be 110. Nike Zoom Fly 5 men's red has a quantity of 2 and a price of 400, so the subtotal should be 800. The expected grand total is 1,410.

If the JSON is valid and the totals match these numbers, then the zero-shot prompt worked well. If the model changes a field name, misses a field, or gives the wrong total, that would show the limitation of relying on instructions without an example.

## Problem 1B: One-Shot Prompt

The one-shot prompt includes one completed example before asking the model to extract information from the second email. In this case, the first email becomes the example, and the second email becomes the new task.

One-shot prompting is useful when the final output needs a specific structure. The example shows the model exactly how the JSON should look. This should make the response more consistent than the zero-shot version.

In [ ]:
one_shot_prompt = f"""
You will extract shoe order information from an email and return ONLY valid JSON.
Do not include Markdown, explanation, or code fences.

Use these shoe prices:
- Nike Air Jordan I: 100
- Converse All-Star: 30
- New Balance 990: 110
- Nike Zoom Fly 5 men's red: 400

Example email:
{email_1}

Example JSON output:
{{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "items": [
    {{
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 100,
      "shoe_subtotal": 200
    }},
    {{
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 30,
      "shoe_subtotal": 300
    }},
    {{
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 110,
      "shoe_subtotal": 110
    }},
    {{
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 400,
      "shoe_subtotal": 800
    }}
  ],
  "grand_total": 1410
}}

Now extract the same type of information from this new email:
{email_2}
"""

one_shot_response = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a careful data extraction assistant. Return valid JSON only."
        },
        {
            "role": "user",
            "content": one_shot_prompt
        }
    ],
    temperature=0
)

one_shot_json = json.loads(one_shot_response.choices[0].message.content)
print(json.dumps(one_shot_json, indent=2))

## One-Shot Result Commentary

For the second email, the expected shoe agent name is Michelle because the email begins with “Hey Michelle.” The expected customer is Artis Gilmore, and the address should be separated into street, city, state, and ZIP code.

The important part is that the model should copy the structure of the example but not copy the values from the example. The quantities are different in the second email. Nike Air Jordan I has a quantity of 1, Converse All-Star has a quantity of 20, New Balance 990 has a quantity of 2, and Nike Zoom Fly 5 men's red has a quantity of 5. Using the same prices, the subtotals should be 100, 600, 220, and 2,000. The expected grand total is 2,920.

A strong one-shot result should look almost identical in structure to the example JSON while accurately reflecting the new customer and order details.

In [ ]:
# Optional validation checks for Problem 1.
# These checks help confirm that the model returned the expected totals.

expected_zero_total = 1410
expected_one_total = 2920

print("Zero-shot grand total:", zero_shot_json.get("grand_total"))
print("Zero-shot total correct:", zero_shot_json.get("grand_total") == expected_zero_total)

print("One-shot grand total:", one_shot_json.get("grand_total"))
print("One-shot total correct:", one_shot_json.get("grand_total") == expected_one_total)

## Problem 1 Comparison: Zero-Shot vs. One-Shot Prompting

The zero-shot and one-shot prompts are similar because both ask the model to extract customer information, extract product information, assign shoe prices, calculate subtotals, and return a grand total. Both responses should be structured as JSON, which makes the output easier to check and reuse in another program.

The main difference is the amount of guidance. The zero-shot prompt only explains the task. The one-shot prompt shows the task plus a completed example. Because of that, the one-shot prompt gives the model a stronger pattern to follow.

In my evaluation, the one-shot approach is stronger for this assignment because the required output format is very specific. The example reduces the chance that the model will rename fields, leave out fields, or format the item list differently. The zero-shot prompt can still work, but it depends more heavily on the clarity of the written instructions.

# Problem 2: Chat Completion for Textbook-Style Math Prompting

Problem 2 recreates the math chat from the textbook example using OpenAI. The prompt gives the model several question-and-answer examples before asking a new question. This is similar to few-shot prompting because the model receives examples of the desired response style.

The final question is: “When I was 6 my sister was half my age. Now I'm 70 how old is my sister?” The correct answer is 67 because when the speaker was 6, the sister was 3. That means the sister is 3 years younger, and the age difference stays the same.

In [ ]:
math_prompt = """
Q: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?
A: There are 4 days from Monday to Thursday. 5 computers were added each day. That means in total 4 * 5 = 20 computers were added. There were 9 computers initially, so now there are 9 + 20 = 29 computers. The answer is 29.

Q: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?
A: Michael initially had 58 balls. He lost 23 on Tuesday, so after that he has 58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answer is 33.

Q: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?
A: She bought 5 bagels for $3 each. This means she spent $15. She has $8 left.

Q: When I was 6 my sister was half my age. Now I'm 70 how old is my sister?
A:
"""

math_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Answer math word problems clearly and briefly, following the examples provided by the user."
        },
        {
            "role": "user",
            "content": math_prompt
        }
    ],
    temperature=0
)

print(math_response.choices[0].message.content)

## Problem 2 Result Commentary

The response to Problem 2 should identify that the sister was 3 years old when the speaker was 6. The key point is that the sister was not always half the speaker’s age. Instead, she was 3 years younger at that time. Since age differences stay constant, the sister should be 67 when the speaker is 70.

The earlier examples use a simple explanation style. They identify the starting number, apply the change, and then state the final answer. The model should follow the same pattern for the sister question by identifying the original age relationship, finding the age difference, and applying that difference to the current age.

A wrong answer would be 35 because that would apply “half my age” to the current age of 70. That interpretation ignores the wording “When I was 6,” so it would not be logically correct.

## Explicit Comparison of Replies Across Both Problems

Problem 1 and Problem 2 are similar because both rely on prompt design to guide the model. In Problem 1, the prompt guides the model toward a structured JSON response. In Problem 2, the examples guide the model toward a short math explanation. In both problems, the quality of the response depends on how clearly the expected output is shown.

They are different because Problem 1 is mainly an extraction and formatting task, while Problem 2 is mainly a reasoning task. Problem 1 can be checked by confirming field names, quantities, prices, subtotals, and the grand total. Problem 2 can be checked by confirming whether the model understands the age relationship correctly.

The zero-shot response in Problem 1 depends on direct instructions. The one-shot response in Problem 1 depends on both instructions and a sample answer. The Problem 2 response also depends on examples, but instead of copying JSON structure, the model copies the explanation style. This shows that examples can guide both formatting tasks and natural language reasoning tasks.

Overall, the one-shot extraction response and the math response are more similar to each other than the zero-shot response because both use examples to teach the model what kind of answer is expected. However, their final outputs are very different: one is machine-readable JSON, and the other is a human-readable explanation.

# Final Reflection

This exercise showed me that prompt engineering is important when using a model through an API. The model can extract information, calculate totals, and answer math word problems, but the quality of the result depends on the prompt.

For structured extraction, I found the one-shot method more reliable because the example gave the model a clear format to follow. For the math problem, the examples helped establish the style of response and encouraged the model to explain the answer in a similar way.

I also learned that Markdown in a notebook should do more than explain the code. The code already shows what is being run. The Markdown should explain why the prompt was designed that way, what results were expected, and how the response should be evaluated. That is why I included commentary comparing the prompt types and explaining how I checked the results.

# References

Neugebauer, F. (2026). *Week 4 exercise instructions*. DSC670 Advanced Uses of Generative AI, Bellevue University.

OpenAI. (2026). *OpenAI API documentation*. https://platform.openai.com/docs

OpenAI. (2026). *Chat completions API reference*. https://platform.openai.com/docs/api-reference/chat/create

Project Jupyter. (2026). *Jupyter Notebook documentation*. https://docs.jupyter.org/

Python Software Foundation. (2026). *Python documentation*. https://docs.python.org/3/